# Run: simulated ChArUco calibration vs the true intrinsics

The sim camera is an ideal pinhole with known K (from the SDF, `data_generator.true_intrinsics`).
Calibrating it from simulated ChArUco images tests the calibration code (gambit_vision M1) where
the right answer is known; the same code then runs on the real camera.

**Deliverable:** "Gazebo world, simulated calibration compared against the true intrinsics" (Updated plan).
Also the fallback under the cut rule (Gazebo stays calibration-only).

**TODO**
- [ ] add a ChArUco board model to the sim (texture from `cv2.aruco.CharucoBoard.generateImage`)
- [ ] teleport it to ~20 poses (tilts 0..45 deg, all image regions), grab a frame at each
- [ ] detect + calibrate with gambit_vision's calibration code; compare with `compare_intrinsics`
- [ ] pass criterion (proposed): |fx error| < 1 %, principal point within 5 px, small distortion

In [ ]:
%run ./data_generator.ipynb
import cv2

K_TRUE = true_intrinsics()
print(K_TRUE)
# Board spec: keep in sync with the printed real board (gambit_vision M1 config).
SQUARES_X, SQUARES_Y, SQUARE_M, MARKER_M = 7, 5, 0.03, 0.022
DICT = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
charuco = cv2.aruco.CharucoBoard((SQUARES_X, SQUARES_Y), SQUARE_M, MARKER_M, DICT)

## Offline dry run (no Gazebo)
Project the ChArUco corners with the true K from random poses and calibrate: this checks the
pipeline math, and gives the best-case error to expect from the sim.

In [ ]:
rng = np.random.default_rng(0)
obj = charuco.getChessboardCorners()
obj_pts, img_pts = [], []
for _ in range(20):
    rvec = rng.normal(scale=0.35, size=3).reshape(3, 1)
    tvec = np.array([[rng.uniform(-0.05, 0.05)], [rng.uniform(-0.05, 0.05)], [rng.uniform(0.35, 0.5)]]) \
        - np.array([[SQUARES_X * SQUARE_M / 2], [SQUARES_Y * SQUARE_M / 2], [0]])
    uv, _ = cv2.projectPoints(obj, rvec, tvec, K_TRUE, np.zeros(5))
    uv += rng.normal(scale=0.2, size=uv.shape)          # sub-pixel detection noise
    obj_pts.append(obj.astype(np.float32)); img_pts.append(uv.astype(np.float32))
rms, K_est, dist, _, _ = cv2.calibrateCamera(obj_pts, img_pts, (640, 480), None, None)
print("rms", rms, compare_intrinsics(K_est, K_TRUE, dist))

## In the sim (TODO)

In [ ]:
def capture_charuco_views(n_views, backend):
    """TODO: for each of n_views random board poses: backend.send(SetPose('charuco_board', ...)),
    frame = grab_frame(), keep frames where >= 50 % of the corners are detected."""
    raise NotImplementedError("TODO: capture_charuco_views")

frames = capture_charuco_views(25, GzCliBackend())
# TODO: K_est, dist = <gambit_vision calibration>(frames, charuco); print(compare_intrinsics(K_est, K_TRUE, dist))